In [25]:
import torch
from transformers import BertTokenizer, RobertaTokenizer, LongformerTokenizer, AutoTokenizer, RobertaModel, LongformerModel, BertModel, AutoModel
from neuralNetwork import NeuralNetwork
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim import AdamW
import random
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from helper import Dataset
import json
import os
from helper import dict_lists_to_list_of_dicts

In [4]:
class Model(NeuralNetwork):

    def __init__(self, bert_type, out_shape, dropout = .1) -> None:
        super().__init__()

        if "FacebookAI/roberta-base" == bert_type:
            self.bert =  RobertaModel.from_pretrained(bert_type)
        elif "bert-base-uncased" == bert_type:
            self.bert = BertModel.from_pretrained(bert_type)
        elif "allenai/longformer-base-4096" == bert_type:
            self.bert = LongformerModel.from_pretrained(bert_type)
        elif "microsoft/codebert-base" == bert_type:
            self.bert = AutoModel.from_pretrained(bert_type)
        else:
            self.bert = AutoModel.from_pretrained(bert_type)

        self.drop_out = nn.Dropout(dropout)
        self.linear_out = nn.Linear(self.bert.config.hidden_size, out_shape)

    def forward(self, inputs):
        _, out = self.bert(**inputs, return_dict = False)
        out = self.drop_out(out)
        return self.linear_out(out)

In [9]:
def train_validation_test_split(x, y, test_buckets = []):
    BUCKETS = 10

    N_ELEMENTS = len(x)

    BUCKET_SIZE = N_ELEMENTS // BUCKETS

    TEST_BUCKETS = 1


    x_local = x.copy()
    y_local = y.copy()
    x_test, y_test = [], []
    
    if len(test_buckets) == 0: 
        for _ in range(TEST_BUCKETS):
            idx = random.randint(0, BUCKETS)
            while idx in test_buckets:
                idx = random.randint(0, BUCKETS)
            test_buckets.append(idx)

    for bucket in test_buckets:
        idx = bucket * BUCKET_SIZE
        for _ in range(BUCKET_SIZE):
            x_test.append(x_local.pop(idx))
            y_test.append(y_local.pop(idx))

    train_elements = (len(y_local) // 10) * 9
    x_train = x_local[:train_elements]
    y_train = y_local[:train_elements]

    x_validation = x_local[train_elements:]
    y_validation = y_local[train_elements:]
    
    return x_train, y_train, x_validation, y_validation, x_test, y_test


def get_tokenizer(bert_type:str):
    if "FacebookAI/roberta-base" == bert_type:
        return RobertaTokenizer.from_pretrained(bert_type)
    elif "bert-base-uncased" == bert_type:
        return BertTokenizer.from_pretrained(bert_type)
    elif "allenai/longformer-base-4096" == bert_type:
        return LongformerTokenizer.from_pretrained(bert_type)
    elif "microsoft/codebert-base" == bert_type:
        return AutoTokenizer.from_pretrained(bert_type)
    elif "../weights/tokenizer_weights" == bert_type:
        return AutoTokenizer.from_pretrained(bert_type)
    else:
        raise Exception("bert type unrecognised")

In [8]:
f = open("../data/datasets/dataset_CarSequencing-2024-03-19.json")
dataset = json.load(f)
f.close()

In [27]:
def get_dataset_split(dataset:list[dict], tokenizer) -> tuple[Dataset, Dataset, Dataset]:
    BIN_TYPE = "avg_bin"
    BINS = 10

    x, y = [], []
    for datapoint in dataset:
        x.append(datapoint["instance_value"])
        y_val = torch.zeros(BINS)
        y_val[datapoint[BIN_TYPE]] = 1
        y.append(y_val)

    x = dict_lists_to_list_of_dicts(tokenizer(x, padding=True, truncation=True, return_tensors='pt'))

    x_train, y_train, x_validation, y_validation, x_test, y_test = train_validation_test_split(x, y, [9])

    return Dataset(x_train, y_train), Dataset(x_validation, y_validation), Dataset(x_test, y_test)
    

In [17]:
tokenizer = get_tokenizer("../weights/tokenizer_weights")

In [29]:
train_dataset, validation_dataset, test_dataset = get_dataset_split(dataset, tokenizer)
train_data_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
validation_data_loader = DataLoader(validation_dataset, batch_size=4)
test_data_loader = DataLoader(test_dataset, batch_size=4)

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [22]:
bert_type = "microsoft/codebert-base"
model = Model(bert_type, 10)

In [30]:
extraction_function = lambda x: torch.max(x, -1)[1].cpu()
f1_metric = lambda y_true, y_pred: f1_score(y_true, y_pred, average="macro")

model.train_network(train_data_loader, validation_data_loader,
                    learning_rate=1e-2,
                    epochs=5,
                    device=device,
                    metrics={
                        "accuracy": accuracy_score,
                        "f1": f1_metric
                    },
                    output_extraction_function=extraction_function,
                    batch_size=32,
                    verbose=True)

EPOCH 1 training loss:      5.545 - validation loss:      3.551                                                                                                                                                                                                                                          
EPOCH 1 training accuracy:      0.974 - validation accuracy:      0.976
EPOCH 1 training f1:      0.943 - validation f1:      0.946
----------------------------------------------------------------------------------------------------

EPOCH 2 training loss:      2.889 - validation loss:      2.873                                                                                                                                                                                                                                          
EPOCH 2 training accuracy:      0.974 - validation accuracy:      0.976
EPOCH 2 training f1:      0.942 - validation f1:      0.946
--------------------------------------

({'accuracy': [0.9740087040618955,
   0.9740087040618955,
   0.9740087040618955,
   0.9740087040618955,
   0.9740087040618955],
  'f1': [0.942517884007242,
   0.9415814681772081,
   0.9433161401246462,
   0.9427481501949545,
   0.9420443032145122],
  'loss': [tensor(5.5448, device='cuda:0'),
   tensor(2.8891, device='cuda:0'),
   tensor(3.7627, device='cuda:0'),
   tensor(3.6955, device='cuda:0'),
   tensor(5.3287, device='cuda:0')]},
 {'accuracy': [0.9761904761904762,
   0.9761904761904762,
   0.9761904761904762,
   0.9761904761904762,
   0.9761904761904762],
  'f1': [0.9455782312925166,
   0.9455782312925166,
   0.9455782312925166,
   0.9455782312925166,
   0.9455782312925166],
  'loss': [tensor(3.5515, device='cuda:0'),
   tensor(2.8726, device='cuda:0'),
   tensor(3.2829, device='cuda:0'),
   tensor(4.0093, device='cuda:0'),
   tensor(4.6336, device='cuda:0')]})

In [31]:
model.bert.save_pretrained("../weights/bert")